In [11]:
import re
from pathlib import Path
import string
from functools import reduce
from math import log
import itertools
import nltk
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [2]:
# Enter smoothing or no smoothing.
smoothing = 1
filename = "textfile.txt"

TASK 1 (A): Pre-Processing of RAW Text File: Split File to Sentences

In [3]:
# Loads file
# input - filename.txt
# returns a list of sentences seperated by newline in the textfile.
def load_file(filename):
    with open(filename, 'r') as file:
        lines = file.read().split('\n')
    return lines

TASK 1 (B): Perform Tokenization Technique on Raw Text File

In [9]:
# Tokenizes the sentences meaning split the sentences into words seperated by the "white sapce".
# input - List of sentences
# returns a list of lists of each sentence being tokenized.
import nltk
from nltk.tokenize import word_tokenize

def tokenize_sentence(lines):
    tokenized_sentences = []
    for line in lines:
        tokens = word_tokenize(line)
        tokenized_sentences.append(tokens)
    return tokenized_sentences

In [12]:
dataset = load_file(filename)
dataset = tokenize_sentence(dataset)
dataset = prep_data(dataset)

No of sentences in Corpus: 10060


TASK 1 (B): Perform Removing of StopWords and Empty Strings, Stemming, Lemmatization and Appending Technique on Raw Text File

In [5]:
# remove punctuations -print(string.punctuation) ---- !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~ ----
# remove empty strings.
# lower case all the words
# add <s> at the beginning and </s> at the end of every sentence in the corpus.
# input - list of lists of words obtained from "tokenize_sentence" function.
# returns - list of lists
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

def prep_data(lines):
    stopWords = set(stopwords.words('english'))
    ps = PorterStemmer()
    preprocessed_data = []

    for sentence in lines:
        # Remove punctuation, empty strings, and convert to lowercase
        cleaned_sentence = [word.lower() for word in sentence if word not in string.punctuation and word not in ['', ' ']]

        # Stem words
        stemmed_sentence = [ps.stem(word) for word in cleaned_sentence]

        # Append <s> at the beginning and </s> at the end
        final_sentence = ['<s>'] + stemmed_sentence + ['</s>']

        preprocessed_data.append(final_sentence)

    print("No of sentences in Corpus: " + str(len(preprocessed_data)))
    return preprocessed_data

In [13]:
# Counts the no. of times a word repeats (frequency of each word) in the corpus.
# input - list of lists of words obtained from "prep_data"
# returns - a dictionary defined as {word:frequency} for words of the corpus including <s> and </s>.
def freq_of_unique_words(lines):
    word_frequency = {}  # Initialize an empty dictionary to store word frequencies

    for sentence in lines:
        for word in sentence:
            if word in word_frequency:
                word_frequency[word] += 1  # Increment the count for an existing word
            else:
                word_frequency[word] = 1  # Initialize the count for a new word

    unique_word_count = len(word_frequency)  # Calculate the total number of unique words
    print("No of unique words in corpus: " + str(unique_word_count))

    return word_frequency  # Return the dictionary of word frequencies

In [14]:
unique_word_frequency = freq_of_unique_words(dataset)
#len(unique_word_frequency)

No of unique words in corpus: 10843


TASK2: Implementation of Naive Baye's Classifier from Scratch

In [15]:
import numpy as np

class NaiveBayesClassifier:
    def __init__(self):
        self.class_priors = {}  # Dictionary to store class priors
        self.class_stats = {}   # Dictionary to store class statistics (mean and variance)

    def calc_prior(self, target):
        # Calculate prior probabilities P(y)
        total_samples = len(target)
        unique_classes, class_counts = np.unique(target, return_counts=True)

        for c, count in zip(unique_classes, class_counts):
            self.class_priors[c] = count / total_samples

    def calc_statistics(self, features, target):
        # Calculate mean and variance for each class and feature
        unique_classes = np.unique(target)

        for c in unique_classes:
            # Select data points belonging to class c
            class_data = features[target == c]

            # Calculate mean and variance for each feature and store in a dictionary
            class_stats = {
                "mean": np.mean(class_data, axis=0),
                "variance": np.var(class_data, axis=0)
            }

            self.class_stats[c] = class_stats

    def gaussian_density(self, x, mean, variance):
        # Calculate probability using Gaussian density function
        exponent = np.exp(-(x - mean)**2 / (2 * variance))
        prob = (1 / (np.sqrt(2 * np.pi * variance))) * exponent
        return prob

    def calc_posterior(self, x):
        posteriors = {}

        for c in self.class_priors.keys():
            prior = self.class_priors[c]
            likelihood = 1.0  # Initialize likelihood

            for i, feature in enumerate(x):
                mean = self.class_stats[c]["mean"][i]
                variance = self.class_stats[c]["variance"][i]

                # Calculate likelihood using Gaussian density
                likelihood *= self.gaussian_density(feature, mean, variance)

            posterior = prior * likelihood
            posteriors[c] = posterior

        # Return the class with the highest posterior probability
        predicted_class = max(posteriors, key=posteriors.get)
        return predicted_class


TASK: Train the Model on Given Classification Dataset by splitting it into ratio of 80:20 and calculate the accuracy of the trained classifier.

In [27]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OneHotEncoder

# Load the dataset
data = pd.read_csv('adult.data', header=None)

# Rename the columns to meaningful names (assuming these are the column names)
data.columns = ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status',
                'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss',
                'hours_per_week', 'native_country', 'income']

# Assuming your target variable is 'income' and features include other columns
X = data.drop('income', axis=1)  # Features
y = data['income']  # Target variable

# Encode categorical features using one-hot encoding
categorical_cols = ['workclass', 'education', 'marital_status', 'occupation',
                    'relationship', 'race', 'sex', 'native_country']

encoder = OneHotEncoder(sparse=False, drop='first')
X_encoded = encoder.fit_transform(X[categorical_cols])

# Get the feature names after one-hot encoding
feature_names = encoder.get_feature_names_out(input_features=categorical_cols)

X_encoded_df = pd.DataFrame(X_encoded, columns=feature_names)

# Drop the original categorical columns and concatenate the encoded columns
X.drop(categorical_cols, axis=1, inplace=True)
X = pd.concat([X, X_encoded_df], axis=1)

# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create an instance of the Gaussian Naive Bayes classifier
nb_classifier = GaussianNB()

# Train the classifier on the training data
nb_classifier.fit(X_train, y_train)

# Make predictions on the testing data
y_pred = nb_classifier.predict(X_test)

# Calculate the accuracy of the model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Accuracy: 0.7990173499155535


TASK 3: Implement Naive Bayes Classifier Using Built-In function
Calculate the Accuracy Score and Metrics for Given Dataset and Compare the Results with your designed algorithm.
Compare using classification matrix and Plots for accuracy, precision and Recall

In [28]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import GaussianNB

# Load your dataset into X and y

# Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create an instance of the Gaussian Naive Bayes classifier from scikit-learn
nb_classifier = GaussianNB()

# Train the classifier on the training data
nb_classifier.fit(X_train, y_train)

# Make predictions on the testing data
y_pred = nb_classifier.predict(X_test)

# Calculate the accuracy using scikit-learn's accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy (Scikit-Learn):", accuracy)


Accuracy (Scikit-Learn): 0.7990173499155535
